**SETUP**

In [ ]:
# @title
# Install dependencies
!pip install biopython pandas requests tqdm

In [ ]:
# @title
# Download signal peptides list

import requests, gzip, shutil, os

QUERY  = "reviewed:true"
FIELDS = "accession,ft_signal,organism_id"
BASE   = "https://rest.uniprot.org/uniprotkb"
s = requests.Session()

# Validate field names + capture release provenance. A bad field name errors here.
r = s.get(f"{BASE}/search",
          params={"query": QUERY, "fields": FIELDS, "format": "tsv", "size": 1},
          timeout=60)
r.raise_for_status()
total   = r.headers.get("X-Total-Results")
release = r.headers.get("X-UniProt-Release")
rdate   = r.headers.get("X-UniProt-Release-Date")
print(f"release {release} ({rdate})   expected rows: {total}")
print("HEADER:", r.text.splitlines()[0])
print("SAMPLE:", r.text.splitlines()[1][:200])

# Stream the full set to disk.
with s.get(f"{BASE}/stream",
           params={"query": QUERY, "fields": FIELDS,
                   "format": "tsv", "compressed": "true"},
           stream=True, timeout=(30, 1800)) as resp:
    resp.raise_for_status()
    with open("swissprot.tsv.gz", "wb") as f:
        for chunk in resp.iter_content(1 << 20):
            f.write(chunk)

with gzip.open("swissprot.tsv.gz", "rb") as fin, open("swissprot.tsv", "wb") as fout:
    shutil.copyfileobj(fin, fout)

# Verify completeness.
n = sum(1 for _ in open("swissprot.tsv", encoding="utf-8")) - 1
print(f"\n{n} data rows | {os.path.getsize('swissprot.tsv')/1e6:.1f} MB")
print("MATCH" if str(n) == total else f"MISMATCH vs X-Total-Results ({total}) — incomplete")

**FASTA Analyser**

Upload a FASTA file


In [ ]:
# @title
import pandas as pd
import re
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm import tqdm


INPUT_FASTA = "input.fasta"
SIGNAL_FILE = "swissprot.tsv"
OUTPUT_TSV = "output.tsv"

signal_df = pd.read_csv(SIGNAL_FILE, sep="\t", dtype=str)

signal_dict = {}

for _, row in signal_df.iterrows():

    entry = str(row["Entry"]).strip()

    signal_info = row["Signal peptide"]

    organism_id = row.get("Organism (ID)", "")

    signal_length = None

    if pd.notna(signal_info) and str(signal_info).strip() != "":

        m = re.search(r"SIGNAL\s+1\.\.(\d+)", str(signal_info))

        if m:
            signal_length = int(m.group(1))

    signal_dict[entry] = {
        "signal_length": signal_length,
        "organism_id": organism_id
    }


def clean_sequence(seq):
    """Keep only the 20 standard amino acids."""
    return re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", str(seq).upper())


def extract_uniprot_id(header):
    """
    Extract UniProt accession from common FASTA headers.

    Handles:
    >sp|P12345|...
    >tr|Q9ABC1|...
    >P12345 ...
    """

    m = re.search(r"\|([A-Z0-9]+)\|", header)
    if m:
        return m.group(1)

    m = re.match(r"^([A-Z0-9]+)", header)
    if m:
        return m.group(1)

    return None


def compute_properties(seq):

    if len(seq) == 0:
        return None

    analysis = ProteinAnalysis(seq)

    aa_freq = analysis.amino_acids_percent

    result = {
        "length": len(seq),
        "mw": analysis.molecular_weight(),
        "gravy": analysis.gravy(),
        "pI": analysis.isoelectric_point(),
    }

    for aa in "ACDEFGHIKLMNPQRSTVWY":
        result[f"freq_{aa}"] = aa_freq.get(aa, 0)

    return result


rows = []

records = list(SeqIO.parse(INPUT_FASTA, "fasta"))

for record in tqdm(records):

    header = record.id
    description = record.description

    sequence = clean_sequence(record.seq)

    uniprot = extract_uniprot_id(description)

    signal_present = False
    signal_length = 0
    organism_id = ""
    signal_source = "not_found"

    if uniprot in signal_dict:

        organism_id = signal_dict[uniprot]["organism_id"]

        if signal_dict[uniprot]["signal_length"] is not None:

            signal_present = True
            signal_length = signal_dict[uniprot]["signal_length"]
            signal_source = "sp.tsv"

        else:
            signal_source = "sp.tsv_no_signal"

    if signal_present and signal_length < len(sequence):
        trimmed_sequence = sequence[signal_length:]
    else:
        trimmed_sequence = sequence

    full_props = compute_properties(sequence)
    trimmed_props = compute_properties(trimmed_sequence)

    row = {
        "header": header,
        "uniprot_id": uniprot,
        "organism_id": organism_id,
        "sequence": sequence,
        "signal_peptide": signal_present,
        "signal_length": signal_length,
        "signal_source": signal_source,
        "trimmed_sequence": trimmed_sequence,
    }

    if full_props:
        for k, v in full_props.items():
            row[f"full_{k}"] = v

    if trimmed_props:
        for k, v in trimmed_props.items():
            row[f"trim_{k}"] = v

    rows.append(row)


output = pd.DataFrame(rows)

output.to_csv(
    OUTPUT_TSV,
    sep="\t",
    index=False
)

print(f"\nDone! Results saved to {OUTPUT_TSV}")
print(f"Processed {len(output)} proteins.")

**MaxQuant Analyser**

Upload evidence.txt into the session drive

In [ ]:
# @title

import pandas as pd
import re
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm import tqdm


INPUT_FILE = "evidence.txt"
OUTPUT_TSV = "output.tsv"


def clean_sequence(seq):
    """Keep only standard amino acids (removes modifications)."""
    return re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', str(seq).upper())


def extract_first_uniprot(protein_field):
    """Extract first UniProt ID from Proteins column."""
    if pd.isna(protein_field):
        return None
    return str(protein_field).split(";")[0]


def compute_properties(seq):
    """Compute peptide properties using Biopython."""
    if len(seq) == 0:
        return None

    analysis = ProteinAnalysis(seq)
    aa_freq = analysis.amino_acids_percent  # correct for new Biopython

    return {
        "length": len(seq),
        "mw": analysis.molecular_weight(),
        "gravy": analysis.gravy(),
        "pI": analysis.isoelectric_point(),
        **{f"freq_{aa}": aa_freq.get(aa, 0) for aa in "ACDEFGHIKLMNPQRSTVWY"}
    }


mq = pd.read_csv(INPUT_FILE, sep="\t")

# Keep only relevant columns
mq = mq[["Sequence", "Proteins"]].copy()


rows = []

for _, row in tqdm(mq.iterrows(), total=len(mq)):
    peptide_raw = row["Sequence"]
    proteins_field = row["Proteins"]

    peptide = clean_sequence(peptide_raw)
    uniprot_id = extract_first_uniprot(proteins_field)

    props = compute_properties(peptide)

    out = {
        "Proteins": uniprot_id,
        "Sequence": peptide
    }

    if props:
        for k, v in props.items():
            out[k] = v

    rows.append(out)


df = pd.DataFrame(rows)
df.to_csv(OUTPUT_TSV, sep="\t", index=False)

print("Done! Saved to:", OUTPUT_TSV)

**Spectronaut Analyser**

Upload spectronaut output file with atleast these 2 columns "PG.ProteinGroups", "EG.PrecursorId"

In [ ]:
# @title

import pandas as pd
import re
from collections import Counter
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm import tqdm


INPUT_FILE = "Spectronaut.tsv"
OUTPUT_FILE = "output.tsv"

PROTON_MASS = 1.007276466812


# Common PTMs (Da)

MOD_MASSES = {
    "Carbamidomethyl": 57.021464,
    "Oxidation": 15.994915,
    "Phospho": 79.966331,
    "Acetyl": 42.010565,
    "Deamidated": 0.984016,
    "Pyro-glu": -17.026549,
    "Gln->pyro-Glu": -17.026549,
    "Glu->pyro-Glu": -18.010565,
    "GlyGly": 114.042927,
    "TMT6plex": 229.162932,
    "TMTpro": 304.207146,
    "Label:13C(6)15N(2)": 8.014199,
    "Label:13C(6)15N(4)": 10.008269,
}


ptm_counter = Counter()
unknown_ptm_counter = Counter()



def parse_precursor(precursor):

    precursor = str(precursor)


    # Charge state

    charge_match = re.search(r"\.(\d+)$", precursor)

    charge = None
    if charge_match:
        charge = int(charge_match.group(1))

    precursor_nocharge = re.sub(r"\.\d+$", "", precursor)


    # Remove flanking underscores

    precursor_nocharge = precursor_nocharge.replace("_", "")


    # Find modifications

    raw_mods = re.findall(r"\[([^\]]+)\]", precursor_nocharge)

    modifications = []
    unknown_modifications = []

    mod_mass_delta = 0.0

    for mod in raw_mods:

        # Example:
        # Carbamidomethyl (C)

        mod_name_match = re.match(r"^([^(]+)", mod)

        if mod_name_match:
            mod_name = mod_name_match.group(1).strip()
        else:
            mod_name = mod.strip()

        modifications.append(mod_name)

        ptm_counter[mod_name] += 1

        if mod_name in MOD_MASSES:
            mod_mass_delta += MOD_MASSES[mod_name]
        else:
            unknown_modifications.append(mod_name)
            unknown_ptm_counter[mod_name] += 1


    # Remove PTM annotations

    clean_seq = re.sub(r"\[.*?\]", "", precursor_nocharge)

    clean_seq = re.sub(
        r"[^ACDEFGHIKLMNPQRSTVWY]",
        "",
        clean_seq.upper()
    )

    return {
        "sequence": clean_seq,
        "charge": charge,
        "modifications": modifications,
        "unknown_modifications": unknown_modifications,
        "mod_mass_delta": mod_mass_delta
    }


def compute_properties(seq, mod_mass_delta, charge):

    if len(seq) == 0:
        return None

    analysis = ProteinAnalysis(seq)

    aa_freq = analysis.amino_acids_percent

    canonical_mw = analysis.molecular_weight()

    modified_mw = canonical_mw + mod_mass_delta

    precursor_mz = None

    if charge and charge > 0:
        precursor_mz = (
            modified_mw + charge * PROTON_MASS
        ) / charge

    result = {
        "length": len(seq),
        "mw": canonical_mw,
        "modified_mw": modified_mw,
        "precursor_mz": precursor_mz,
        "gravy": analysis.gravy(),
        "pI": analysis.isoelectric_point(),
    }

    for aa in "ACDEFGHIKLMNPQRSTVWY":
        result[f"freq_{aa}"] = aa_freq.get(aa, 0)

    return result


# Load data


df = pd.read_csv(INPUT_FILE, sep="\t")

required_columns = [
    "PG.ProteinGroups",
    "EG.PrecursorId"
]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Process

rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    protein_groups = row["PG.ProteinGroups"]
    precursor = row["EG.PrecursorId"]

    parsed = parse_precursor(precursor)

    props = compute_properties(
        parsed["sequence"],
        parsed["mod_mass_delta"],
        parsed["charge"]
    )

    out = {
        "PG.ProteinGroups": protein_groups,
        "EG.PrecursorId": precursor,
        "Sequence": parsed["sequence"],
        "Charge": parsed["charge"],
        "Modifications": "; ".join(parsed["modifications"]),
        "Num_Modifications": len(parsed["modifications"]),
        "Unknown_Modifications": "; ".join(parsed["unknown_modifications"]),
        "Num_Unknown_Modifications": len(parsed["unknown_modifications"]),
        "Modification_Mass_Delta": parsed["mod_mass_delta"],
    }

    if props:
        out.update(props)

    rows.append(out)


# Save output


output_df = pd.DataFrame(rows)

output_df.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False
)

print(f"\nOutput written to: {OUTPUT_FILE}")

# PTM summary

print("\n" + "=" * 60)
print("PTM SUMMARY")
print("=" * 60)

if len(ptm_counter) == 0:
    print("No PTMs detected.")
else:
    print("\nKnown/Observed PTMs:")
    for ptm, count in ptm_counter.most_common():
        print(f"{ptm}: {count}")

print("\n" + "-" * 60)
print("UNKNOWN PTMs")
print("-" * 60)

if len(unknown_ptm_counter) == 0:
    print("No unknown PTMs found.")
else:
    for ptm, count in unknown_ptm_counter.most_common():
        print(f"{ptm}: {count}")

print("\nTotal unique PTMs found:", len(ptm_counter))
print("Total unique unknown PTMs:", len(unknown_ptm_counter))